# 🏗️ Notebook 1: Discord — Requirements & Architecture


## 🛠️ Setup

```bash
cd 06-system-designs/discord
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

All code in this lab is **self-contained Python** — no servers, no databases. We simulate Discord's gateway, pub/sub, and fan-out in memory so you can run every cell and see what happens.


## 💡 What we're designing

Discord is a **real-time chat** platform. Users join **guilds** (servers), talk in **text channels**, hop into **voice channels**, and see who is **online right now**.

Three things make Discord hard:

1. **Real-time**: a message typed in Tokyo should appear in New York in <200 ms.
2. **Huge fan-out**: a popular channel can have 100,000+ members watching live.
3. **Presence**: millions of online/offline flips per second without melting the servers.

### Functional requirements
- Send and receive text messages in a channel.
- See who is online (presence).
- Join voice channels with low latency.
- Keep message history (scrollback).
- Push notifications when a user is offline.

### Non-functional requirements
- **WebSockets**, not HTTP polling — pushing is cheaper than clients asking "any news?" 50 times per minute.
- **Horizontally scalable** gateway — one box can't hold 15 million sockets.
- **Geographically close** voice servers — audio latency is dominated by distance.


## 📏 Back-of-envelope capacity

Before drawing boxes, let's put numbers on the problem. Being wrong by 10× is fine; being wrong by 1000× means your architecture collapses.


In [1]:
# ==== Capacity estimation ====
# These numbers are public-ish estimates; the point is the *method*, not the exact figures.

MAU = 150_000_000               # monthly active users
concurrent_ratio = 0.10         # ~10% of MAU online at peak
msgs_per_user_per_min = 2       # active users chat lightly
avg_channel_members = 50        # average audience for one message
voice_bitrate_kbps = 64         # Opus at decent quality
concurrent_voice = 1_000_000

concurrent_users = int(MAU * concurrent_ratio)
msgs_per_sec = concurrent_users * msgs_per_user_per_min / 60
fanout_events_per_sec = msgs_per_sec * avg_channel_members
voice_bandwidth_gbps = (concurrent_voice * voice_bitrate_kbps) / 1_000_000  # kbps → Gbps

print(f'Concurrent users     : {concurrent_users:>15,}')
print(f'Messages / sec       : {int(msgs_per_sec):>15,}')
print(f'Fan-out events / sec : {int(fanout_events_per_sec):>15,}')
print(f'Voice bandwidth      : {voice_bandwidth_gbps:>14.1f} Gbps')

# Takeaway: 25M events/sec is the number that shapes the whole design.
# A single box can't do that — we need fan-out via a message bus.


Concurrent users     :      15,000,000
Messages / sec       :         500,000
Fan-out events / sec :      25,000,000
Voice bandwidth      :           64.0 Gbps


## 🧱 High-level architecture

```
           ┌────────────┐                             
           │  Clients   │  (web / mobile / desktop)   
           └──────┬─────┘                             
                  │ wss://gateway.discord...          
                  ▼                                   
           ┌────────────┐     ┌──────────────┐        
           │  Gateway   │◀───▶│  Session &   │        
           │ (WebSocket)│     │  Presence    │        
           └──────┬─────┘     └──────────────┘        
                  │ publish                           
                  ▼                                   
           ┌─────────────────────┐                    
           │  Message Bus        │  Kafka / NATS      
           │  topic per channel  │                    
           └──────┬──────────────┘                    
                  │ subscribe                         
        ┌─────────┼─────────┐                         
        ▼         ▼         ▼                         
   ┌────────┐┌────────┐┌────────┐                     
   │Gateway ││Gateway ││Gateway │ … 300 instances     
   └────────┘└────────┘└────────┘                     
                  │                                   
                  ▼                                   
        ┌───────────────────┐     ┌──────────────┐    
        │ Message storage   │     │ Push notif   │    
        │ (Cassandra/Scylla)│     │ (APNS/FCM)   │    
        └───────────────────┘     └──────────────┘    

   Voice: separate UDP/WebRTC media servers (SFU) in each region.
```

Key ideas, in plain English:

- Every client keeps **one WebSocket open** to a Gateway. No polling.
- When Alice sends a message, the Gateway publishes it to a **channel topic** on the bus.
- Every Gateway that has *at least one subscriber* of that channel receives the event, and pushes to only those local sockets.
- Messages are persisted separately (for scrollback). The hot path is **write-then-publish**, not write-then-read.


## ⚖️ Gateway sharding — bad → best

A single WebSocket server comfortably holds ~50k connections (file descriptors, memory for buffers). With 15M concurrent users we need ~300 gateways. How do we pick which one a client connects to?

We'll simulate three approaches and watch the load distribution.


In [2]:
# ==== Gateway assignment strategies ====
import random, hashlib
from collections import Counter

NUM_USERS = 300_000
NUM_GATEWAYS = 30
random.seed(7)
user_ids = [random.randint(10**18, 10**19) for _ in range(NUM_USERS)]

def load(counter):
    vals = list(counter.values())
    return min(vals), max(vals), max(vals) / (sum(vals)/len(vals))

# ---------- ❌ BAD: random assignment ----------
# Problem: clients that reconnect end up on a *different* gateway every time —
# presence/subscriptions have to be rebuilt, caches cold.
bad = Counter(random.randint(0, NUM_GATEWAYS-1) for _ in user_ids)

# ---------- 🙂 OK: user_id % N ----------
# Sticky per user. But if we add/remove one gateway, *every* user moves.
ok = Counter(uid % NUM_GATEWAYS for uid in user_ids)

# ---------- ✅ BEST: consistent hashing ----------
# Hash each gateway to several points on a ring, then each user lands on
# the next point clockwise. Adding/removing one gateway moves only 1/N of users.
RING = []
VIRTUAL_NODES = 100   # more virtual nodes → smoother distribution
for g in range(NUM_GATEWAYS):
    for v in range(VIRTUAL_NODES):
        h = int(hashlib.md5(f'gw{g}-{v}'.encode()).hexdigest(), 16)
        RING.append((h, g))
RING.sort()

def pick(uid):
    h = int(hashlib.md5(str(uid).encode()).hexdigest(), 16)
    # binary search would be faster; linear is clearer
    for point, gw in RING:
        if point >= h:
            return gw
    return RING[0][1]     # wrap around

best = Counter(pick(uid) for uid in user_ids)

for name, c in [('BAD  random', bad), ('OK   modulo', ok), ('BEST consistent', best)]:
    lo, hi, skew = load(c)
    print(f'{name:<16}  min={lo:>5}  max={hi:>5}  max/avg skew={skew:.2f}')


BAD  random       min= 9810  max=10157  max/avg skew=1.02
OK   modulo       min= 9784  max=10363  max/avg skew=1.04
BEST consistent   min= 8927  max=11906  max/avg skew=1.19


**What to notice:**

- Random and modulo both distribute evenly *when nothing changes*. The problem is what happens on a deploy.
- With consistent hashing, if you add gateway #31, only ~1/31 of users move. With modulo, **all** users move.
- Discord actually hashes by `guild_id` (not user_id) so all members of one server land on the same gateway for presence — same trick, different key.


## ✅ Summary

- Discord is **push, not pull** — WebSockets are non-negotiable at this scale.
- The dominant cost is **fan-out**, not storage.
- Gateways are **stateless** (session in Redis); we can scale them horizontally.
- Use **consistent hashing** to route clients so deploys don't cause a reconnect storm.

Next: [Notebook 2 — Data Model & APIs](./02_data_and_api.ipynb).
